# Creating the parquet dataset from SQLite tables

In [1]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

!pwd

%load_ext autoreload
%autoreload 2

Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.
/rds/homes/g/gaddcz/Projects/CPRD/examples/data/3_build_fine_tuning_datasets/Study1_T2D/Hypertension/stratified_dataset


In [2]:
import torch
from hydra import compose, initialize
from omegaconf import OmegaConf
import logging
import time
import pickle 

from FastEHR.dataloader import FoundationalDataModule
from FastEHR.database.collector import SQLiteDataCollector

from CPRD.examples.data.study_criteria import t2d_inclusion_method

torch.manual_seed(1337)

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = "cpu"    # if more informative debugging statements are needed
print(f"Using device: {device}.")


Using device: cuda.


In [3]:
# load the configuration file, override any settings 
with initialize(version_base=None, config_path="../../../../../modelling/SurvivEHR/confs", job_name="dataset_creation_notebook"):
    cfg = compose(config_name="config_CompetingRisk11M", overrides=[])

# Create new dataset 
cfg.data.path_to_ds = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_Hypertension/"
print(OmegaConf.to_yaml(cfg))

is_decoder: true
data:
  batch_size: 64
  unk_freq_threshold: 0.0
  min_workers: 12
  global_diagnoses: false
  repeating_events: true
  path_to_db: /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/cprd.db
  path_to_ds: /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_Hypertension/
  meta_information_path: /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle
  subsample_training: null
experiment:
  type: pre-train
  project_name: SurvivEHR
  run_id: ${head.SurvLayer}PreTrain_small_${experiment.seed}
  fine_tune_id: null
  notes: null
  tags: null
  train: true
  test: true
  verbose: true
  seed: 1337
  log: true
  log_dir: /rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/FoundationModelOutput/
  ckpt_dir: /rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/FoundationModelOutput/checkpoints/
fine_tuning:
  fine_tune_outcomes: null
  custo

## Initialise a collector to interact with the database

We need this as we want to filter our practice inclusion to create regional splits

In [4]:
PATH_TO_DB = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/cprd.db"

collector = SQLiteDataCollector(db_path=PATH_TO_DB)


def reduce_by_health_authority(practice_ids, 
                               health_auth,
                               collector):
    collector.connect()

    
    # reduced_list = []
    # for pid in list_of_practice_ids:
    #     cursor.execute(f"""SELECT HEALTH_AUTH FROM static_table WHERE PRACTICE_ID=='{train_practice_ids[0]}' LIMIT 1""")   # 

    # Use parameterized query with tuple expansion
    placeholders_ids = ",".join("?" for _ in practice_ids)
    placeholder_authorities = ",".join("?" for _ in health_auth)
    query = f"""
        SELECT DISTINCT PRACTICE_ID
        FROM static_table
        WHERE PRACTICE_ID IN ({placeholders_ids})
          AND HEALTH_AUTH in ({placeholder_authorities})
    """
    collector.cursor.execute(query, (*practice_ids, *health_auth))
    filtered_ids = [row[0] for row in collector.cursor.fetchall()]

    collector.disconnect()
    return filtered_ids

### Available health authorities

In [5]:
collector.connect()
collector.cursor.execute("""PRAGMA table_info(static_table);""")
columns_info = collector.cursor.fetchall()
print([c[1] for c in columns_info])
collector.disconnect()

['PRACTICE_ID', 'PATIENT_ID', 'ETHNICITY', 'YEAR_OF_BIRTH', 'SEX', 'COUNTRY', 'IMD', 'HEALTH_AUTH', 'INDEX_DATE', 'START_DATE', 'END_DATE']


In [6]:
collector.connect()

query = f"SELECT DISTINCT HEALTH_AUTH FROM static_table;"
collector.cursor.execute(query)

unique_values = [row[0] for row in collector.cursor.fetchall()]

print(f"Unique health authories:")
for val in unique_values:
    print(val)

collector.disconnect()

Unique health authories:
North West
North East
Yorkshire & The Humber
East Midlands
East of England
London
South East
South West
West Midlands
Northern Ireland
None


### We have already specified training-test-validation splits for general practices in the UK

We must ensure we re-use the same splits to avoid data-leakage

In [7]:
# Original pre-training splits
overwrite_practice_ids = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/practice_id_splits.pickle"
with open(overwrite_practice_ids, 'rb') as f:
    splits = pickle.load(f)

train_practice_ids = splits["train"]
val_practice_ids = splits["val"]
test_practice_ids = splits["test"]
print(f"Initial pre-training splits have {len(train_practice_ids)}, "
      f"{len(val_practice_ids)} and {len(test_practice_ids)} "
      f"training, validation and test practices, respectively. "
     )

Initial pre-training splits have 1330, 74 and 74 training, validation and test practices, respectively. 


### We now want to divide this existing training cohort into sub-populations

In [8]:
save_path = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/"

### Create a training split containing only patients from London practices

In [ ]:
# # Create a split for London and the South East
# authority_group_1 = ["London"]

# group1_train_practice_ids = reduce_by_health_authority(
#     train_practice_ids, authority_group_1, collector
# )
# group1_val_practice_ids = reduce_by_health_authority(
#     val_practice_ids, authority_group_1, collector
# )
# group1_test_practice_ids = reduce_by_health_authority(
#     test_practice_ids, authority_group_1, collector
# )
# print(f"{', '.join(authority_group_1)} splits have {len(group1_train_practice_ids)}, "
#       f"{len(group1_val_practice_ids)} and {len(group1_test_practice_ids)} "
#       f"training, validation and test practices, respectively. "
#       )

# split_group1 = {"train": group1_train_practice_ids, "val": group1_val_practice_ids, "test": group1_test_practice_ids}
# with open(save_path + f'practice_id_splits_{"_".join(authority_group_1)}.pickle', 'wb') as handle:
#     pickle.dump(split_group1, handle, protocol=pickle.HIGHEST_PROTOCOL)


### Create a training split containing only patients from North Eastern practices

In [9]:
# Create a split for the North East of England
authority_group_2 = ["North East"]

group2_train_practice_ids = reduce_by_health_authority(
    train_practice_ids, authority_group_2, collector
)
group2_val_practice_ids = reduce_by_health_authority(
    val_practice_ids, authority_group_2, collector
)
group2_test_practice_ids = reduce_by_health_authority(
    test_practice_ids, authority_group_2, collector
)
print(f"{', '.join(authority_group_2)} splits have {len(group2_train_practice_ids)}, "
       f"{len(group2_val_practice_ids)} and {len(group2_test_practice_ids)} "
       f"training, validation and test practices, respectively. "
     )

split_group2 = {"train": group2_train_practice_ids, "val": group2_val_practice_ids, "test": group2_test_practice_ids}
with open(save_path + f'practice_id_splits_{"_".join(authority_group_2)}.pickle', 'wb') as handle:
    pickle.dump(split_group2, handle, protocol=pickle.HIGHEST_PROTOCOL)


North East splits have 50, 2 and 4 training, validation and test practices, respectively. 


### Build the dataset for each of these collections of these general practices splits

In [10]:
# Build for each group 

for auth_group in [authority_group_2]:       #authority_group_1, 

    path_to_ds = save_path + f"Hypertension_{'_'.join(auth_group)}/"
    path_to_split = save_path + f'practice_id_splits_{"_".join(auth_group)}.pickle'
    
    dm = FoundationalDataModule(path_to_db=cfg.data.path_to_db,
                                path_to_ds=path_to_ds,
                                load=False,
                                include_diagnoses=True,
                                include_measurements=True,
                                drop_missing_data=False,
                                drop_empty_dynamic=True,
                                tokenizer="tabular",
                                overwrite_practice_ids=path_to_split,
                                overwrite_meta_information=cfg.data.meta_information_path,
                                study_inclusion_method=t2d_inclusion_method(outcomes=["HYPERTENSION"]),  # min_events=50
                                num_threads=1
                               )

vocab_size = dm.train_set.tokenizer.vocab_size

print(f"{len(dm.train_set)} training patients")
print(f"{len(dm.val_set)} validation patients")
print(f"{len(dm.test_set)} test patients")
print(f"{vocab_size} vocab elements")

INFO:root:Building Polars datasets and saving to /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/Hypertension_North East/
INFO:root:Using train/test/val splits from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/practice_id_splits_North East.pickle
INFO:root:Processing test split...
Thread generating parquet for 4 practices: 100%|██████████| 4/4 [03:39<00:00, 54.88s/it]
INFO:root:Created dataset at /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/Hypertension_North East/split=test with 1696 number of samples
Getting file row counts. This allows the creation of an index to file map, increasing read efficiency: 9it [00:00, 217.65it/s]
INFO:root:	 Obtained with a total of 1696 samples
INFO:root:Processing train split...
Thread generating parquet for 50 practices: 100%|██████████| 50/50 [32:40<00:00, 39.22s/it]  
INFO:root:Created dataset at /rds/projects/g/gokhalkm-optim

17539 training patients
805 validation patients
1696 test patients
265 vocab elements


In [ ]:
dm.train_set.view_sample(1, max_dynamic_events=None, report_time=True)